In [1]:
import neuroglancer
import numpy as np

Create a new (initially empty) viewer.  This starts a webserver in a background thread, which serves a copy of the Neuroglancer client, and which also can serve local volume data and handles sending and receiving Neuroglancer state updates.

In [2]:
viewer = neuroglancer.Viewer()

Print a link to the viewer (only valid while the notebook kernel is running). Note that while the Viewer is running, anyone with the link can obtain any authentication credentials that the neuroglancer Python module obtains. Therefore, be very careful about sharing the link, and keep in mind that sharing the notebook will likely also share viewer links.

In [3]:
viewer

http://127.0.0.1:53916/v/cff787c3fbc8006b794b577631b35f01b7ccf4b6/

Add some example layers using the precomputed data source (HHMI Janelia FlyEM FIB-25 dataset).

In [4]:
with viewer.txn() as s:
  s.layers['image'] = neuroglancer.ImageLayer(source='precomputed://gs://neuroglancer-public-data/flyem_fib-25/image')
  s.layers['segmentation'] = neuroglancer.SegmentationLayer(source='precomputed://gs://neuroglancer-public-data/flyem_fib-25/ground_truth', selected_alpha=0.3)


Display a numpy array as an additional layer.  A reference to the numpy array is kept only as long as the layer remains in the viewer.

Move the viewer position.

In [5]:
with viewer.txn() as s:
    s.voxel_coordinates = [3000.5, 3000.5, 3000.5]

Hide the segmentation layer.

In [6]:
with viewer.txn() as s:
    s.layers['segmentation'].visible = False

In [32]:
# select segment 24114


AttributeError: 'VisibleSegments' object has no attribute 'update'

In [7]:
with viewer.txn() as s:
    s.layout.type = "3d"
    s.layout.cross_sections.visible = False

In [8]:
import copy
import time

def interpolate_to(final_state, frames_per_second=5, seconds=1):
    total_frames = int(round(seconds * frames_per_second))
    initial_state = viewer.state
    for frame_i in range(total_frames):
        t = frame_i / total_frames
        viewer.set_state(
            neuroglancer.ViewerState.interpolate(initial_state, final_state, t)
        )
        time.sleep(1 / frames_per_second)
    viewer.set_state(final_state)

def move_by(offset, **kwargs):
    final_state = copy.deepcopy(viewer.state)
    final_state.voxel_coordinates += offset
    interpolate_to(final_state, **kwargs)

def move_to(final_voxel_coordinates, final_orientation, **kwargs):
    final_state = copy.deepcopy(viewer.state)
    final_state.voxel_coordinates = final_voxel_coordinates
    final_state.crossSectionOrientation = final_orientation
    final_state.projectionOrientation = final_orientation
    interpolate_to(final_state, **kwargs)

def do_move_by():
    move_by([100, 100, 100], seconds=1, frames_per_second=10)

def do_move_to(final_voxel_coordinates, final_orientation, seconds=1, frames_per_second=10):
    move_to(final_voxel_coordinates, final_orientation = final_orientation, seconds=seconds, frames_per_second=frames_per_second)


In [9]:
do_move_to([3474, 3598, 3500], [1, 1, 1, 0], seconds = 1)

In [66]:
with viewer.txn() as s:
    s.crossSectionOrientation = [1, 1, 1, 0.6]

Print the neuroglancer viewer state.  The Neuroglancer Python library provides a set of Python objects that wrap the JSON-encoded viewer state.  `viewer.state` returns a read-only snapshot of the state.  To modify the state, use the `viewer.txn()` function, or `viewer.set_state`.

In [67]:
viewer.state

ViewerState({"dimensions": {"x": [8e-09, "m"], "y": [8e-09, "m"], "z": [8e-09, "m"]}, "position": [3386.277587890625, 3363.4208984375, 3502.6259765625], "crossSectionOrientation": [0.5, 0.5, 0.5, 0.5], "crossSectionScale": 0.38867957090175353, "projectionOrientation": [0.3655225932598114, 0.5771896839141846, 0.6093267202377319, 0.4024503529071808], "projectionScale": 422.383364055973, "layers": [{"type": "image", "source": "precomputed://gs://neuroglancer-public-data/flyem_fib-25/image", "tab": "source", "name": "image"}, {"type": "segmentation", "source": "precomputed://gs://neuroglancer-public-data/flyem_fib-25/ground_truth", "tab": "source", "selectedAlpha": 0.30000000000000004, "segments": ["!24114", "148679"], "name": "segmentation"}], "showSlices": false, "layout": {"type": "3d", "crossSections": {"a": {}}}})

In [68]:
viewer.state.position, viewer.state.projection_orientation


(array([3386.2776, 3363.421 , 3502.626 ], dtype=float32),
 array([0.3655226 , 0.5771897 , 0.6093267 , 0.40245035], dtype=float32))

In [72]:
viewer.state.position, viewer.state.projection_orientation


(array([3420.5, 3363.5, 3502.5], dtype=float32),
 array([0.30634463, 0.6299517 , 0.65337616, 0.2870773 ], dtype=float32))

In [73]:
viewer.state.position, viewer.state.projection_orientation


(array([3429.571 , 3414.8914, 3488.2466], dtype=float32),
 array([0.09664514, 0.7088308 , 0.6953755 , 0.0683486 ], dtype=float32))

In [75]:
# record a series of positions and orientations
df = {}
df['position'] = [[3386.2776, 3363.421 , 3502.626 ], [3420.5, 3363.5, 3502.5], [3429.571 , 3414.8914, 3488.2466]]
df['orientation'] = [[0.3655226 , 0.5771897 , 0.6093267 , 0.40245035], [0.30634463, 0.6299517 , 0.65337616, 0.2870773 ], [0.09664514, 0.7088308 , 0.6953755 , 0.0683486 ]]
df

{'position': [[3386.2776, 3363.421, 3502.626],
  [3420.5, 3363.5, 3502.5],
  [3429.571, 3414.8914, 3488.2466]],
 'orientation': [[0.3655226, 0.5771897, 0.6093267, 0.40245035],
  [0.30634463, 0.6299517, 0.65337616, 0.2870773],
  [0.09664514, 0.7088308, 0.6953755, 0.0683486]]}

In [93]:
# record a series of positions and orientations from viewer.state

df = {}
df['position'] = []
df['orientation'] = []



{'position': [array([2136.3438, 2404.0303, 5106.7476], dtype=float32)],
 'orientation': [array([ 0.04545097, -0.95438886, -0.09901075, -0.27798018], dtype=float32)]}

In [149]:
df['position'].append(viewer.state.position)
df['orientation'].append(viewer.state.projection_orientation)
df

{'position': [array([2136.3438, 2404.0303, 5106.7476], dtype=float32),
  array([2160.0063, 2415.743 , 5059.1445], dtype=float32),
  array([2191.1133, 2426.4768, 5016.5645], dtype=float32),
  array([2229.6423, 2419.184 , 4959.953 ], dtype=float32),
  array([2249.4084, 2435.5754, 4925.8203], dtype=float32),
  array([2272.648 , 2460.7588, 4905.136 ], dtype=float32),
  array([2327.665 , 2469.3704, 4875.121 ], dtype=float32),
  array([2360.0535, 2477.8591, 4848.864 ], dtype=float32),
  array([2388.381 , 2488.5933, 4804.556 ], dtype=float32),
  array([2410.6858, 2492.0818, 4760.704 ], dtype=float32),
  array([2435.7058, 2501.5693, 4711.865 ], dtype=float32),
  array([2455.0706, 2497.957 , 4667.8984], dtype=float32),
  array([2456.854 , 2478.77  , 4635.9443], dtype=float32),
  array([2444.041 , 2476.0317, 4597.9478], dtype=float32),
  array([2427.5562, 2478.7007, 4555.351 ], dtype=float32),
  array([2437.4624, 2488.7664, 4529.302 ], dtype=float32),
  array([2452.4614, 2487.8774, 4500.2603], d

In [157]:
# set the initial state
with viewer.txn() as s:
    s.voxel_coordinates = df['position'][0]
    s.crossSectionOrientation = df['orientation'][0]
    s.projectionOrientation = df['orientation'][0]

In [158]:
# do move to according to the list of positions and orientations in df
for i in range(len(df['position'])):
    do_move_to(df['position'][i], df['orientation'][i], seconds = 0.5)

Print the set of selected segments.|

In [21]:
viewer.state.layers['segmentation'].segments

VisibleSegments([])

Update the state by calling `set_state` directly.

In [22]:
import copy

new_state = copy.deepcopy(viewer.state)
new_state.layers['segmentation'].segments.add(10625)
viewer.set_state(new_state)

'a98cd10c9f7440319adff385fc20269b391d1e6c'

Bind the 't' key in neuroglancer to a Python action.

In [23]:
num_actions = 0
def my_action(s):
    global num_actions
    num_actions += 1
    with viewer.config_state.txn() as st:
      st.status_messages['hello'] = ('Got action %d: mouse position = %r' %
                                     (num_actions, s.mouse_voxel_coordinates))
    print('Got my-action')
    print(f'  Mouse position: {s.mouse_voxel_coordinates}')
    print(f'  Layer selected values: {s.selected_values}')
viewer.actions.add('my-action', my_action)
with viewer.config_state.txn() as s:
    s.input_event_bindings.viewer['keyt'] = 'my-action'
    s.status_messages['hello'] = 'Welcome to this example'

Change the view layout to 3-d.

In [24]:
with viewer.txn() as s:
    s.layout = '3d'
    s.projection_scale = 3000

Take a screenshot (useful for creating publication figures, or for generating videos).  While capturing the screenshot, we hide the UI and specify the viewer size so that we get a result independent of the browser size.

In [25]:
from ipywidgets import Image

screenshot = viewer.screenshot(size=[1000, 1000])
screenshot_image = Image(value=screenshot.screenshot.image)
screenshot_image

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x03\xe8\x00\x00\x03\xe8\x08\x06\x00\x00\x00M\xa3\xd4…

Change the view layout to show the segmentation side by side with the image, rather than overlayed.  This can also be done from the UI by dragging and dropping.  The side by side views by default have synchronized position, orientation, and zoom level, but this can be changed.

In [26]:
with viewer.txn() as s:
    s.layout = neuroglancer.row_layout(
        [neuroglancer.LayerGroupViewer(layers=['image', 'overlay']),
         neuroglancer.LayerGroupViewer(layers=['segmentation'])])

Remove the overlay layer.

In [27]:
with viewer.txn() as s:
    s.layout = neuroglancer.row_layout(
        [neuroglancer.LayerGroupViewer(layers=['image']),
         neuroglancer.LayerGroupViewer(layers=['segmentation'])])

Create a publicly sharable URL to the viewer state (only works for external data sources, not layers served from Python).  The Python objects for representing the viewer state (`neuroglancer.ViewerState` and friends) can also be used independently from the interactive Python-tied viewer to create Neuroglancer links.

In [28]:
print(neuroglancer.to_url(viewer.state))

https://neuroglancer-demo.appspot.com#!%7B%22dimensions%22:%7B%22x%22:%5B8e-09,%22m%22%5D,%22y%22:%5B8e-09,%22m%22%5D,%22z%22:%5B8e-09,%22m%22%5D%7D,%22position%22:%5B3000.5,3000.5,3000.5%5D,%22crossSectionScale%22:1,%22projectionScale%22:3000,%22layers%22:%5B%7B%22type%22:%22image%22,%22source%22:%22precomputed://gs://neuroglancer-public-data/flyem_fib-25/image%22,%22tab%22:%22source%22,%22name%22:%22image%22%7D,%7B%22type%22:%22segmentation%22,%22source%22:%22precomputed://gs://neuroglancer-public-data/flyem_fib-25/ground_truth%22,%22tab%22:%22source%22,%22selectedAlpha%22:0.3,%22segments%22:%5B%2210625%22%5D,%22name%22:%22segmentation%22%7D%5D,%22layout%22:%7B%22type%22:%22row%22,%22children%22:%5B%7B%22type%22:%22viewer%22,%22layers%22:%5B%22image%22%5D,%22layout%22:%22xy%22%7D,%7B%22type%22:%22viewer%22,%22layers%22:%5B%22segmentation%22%5D,%22layout%22:%22xy%22%7D%5D%7D%7D


Stop the Neuroglancer web server, which invalidates any existing links to the Python-tied viewer.

In [161]:
neuroglancer.stop()